### Appendix 1 (Question 1)

In [2]:
import math

# ── Parameters ────────────────────────────────────────────────────────────────
S        = 30      # spot price
K        = 30      # strike (at-the-money)
T        = 0.5     # time to expiry (years)
r        = 0.03    # risk-free rate
q        = 0.01    # continuous dividend yield
C_market = 2.5     # observed market price
sigma_0  = 0.5     # initial guess
tol      = 1e-6    # convergence tolerance (6 decimal places)

# ── Helper functions ───────────────────────────────────────────────────────────
def norm_cdf(x):
    """Standard normal CDF."""
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def norm_pdf(x):
    """Standard normal PDF."""
    return math.exp(-0.5 * x**2) / math.sqrt(2 * math.pi)

# ── Black-Scholes price (Merton model, continuous dividends) ──────────────────
def bs_call(sigma):
    """
    Returns the Black-Scholes call price for a dividend-paying asset.
    Since S = K, ln(S/K) = 0 and drops out of d1.
    """
    d1 = ((r - q + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    price = S * math.exp(-q * T) * norm_cdf(d1) - K * math.exp(-r * T) * norm_cdf(d2)
    return price, d1

# ── Vega — the derivative of C with respect to sigma ─────────────────────────
def vega(d1):
    """
    Vega = dC/d(sigma) = S * e^(-qT) * sqrt(T) * phi(d1)
    This is f'(sigma) in Newton's method.
    """
    return S * math.exp(-q * T) * math.sqrt(T) * norm_pdf(d1)

# ── Newton's method ────────────────────────────────────────────────────────────
def implied_vol_newton(sigma_0, tol=1e-6, max_iter=100):
    """
    Find sigma such that bs_call(sigma) = C_market.
    Solves f(sigma) = bs_call(sigma) - C_market = 0
    using the update: sigma_new = sigma - f(sigma) / f'(sigma)
    """
    sigma = sigma_0
    print(f"{'Iter':>4}  {'sigma':>12}  {'C(sigma)':>12}  {'f = C - 2.5':>13}  {'Vega':>10}")
    print("-" * 60)

    for i in range(max_iter):
        C, d1   = bs_call(sigma)
        v       = vega(d1)
        f       = C - C_market              # function value
        print(f"{i:>4}  {sigma:>12.8f}  {C:>12.8f}  {f:>+13.8f}  {v:>10.6f}")

        sigma_new = sigma - f / v           # Newton step

        if abs(sigma_new - sigma) < tol:    # converged
            sigma = sigma_new
            C, d1 = bs_call(sigma)
            f     = C - C_market
            v     = vega(d1)
            print(f"{i+1:>4}  {sigma:>12.8f}  {C:>12.8f}  {f:>+13.8f}  {v:>10.6f}")
            print(f"\nConverged in {i+1} iterations.")
            return sigma

        sigma = sigma_new

    print("Warning: did not converge within max iterations.")
    return sigma

# ── Run ────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    iv = implied_vol_newton(sigma_0, tol)
    print(f"Implied volatility = {iv:.6f}")


Iter         sigma      C(sigma)    f = C - 2.5        Vega
------------------------------------------------------------
   0    0.50000000    4.31781089    +1.81781089    8.245439
   1    0.27953742    2.48985468    -0.01014532    8.327154
   2    0.28075576    2.49999984    -0.00000016    8.326891
   3    0.28075578    2.50000000    -0.00000000    8.326891

Converged in 3 iterations.
Implied volatility = 0.280756


### Appendix 2 (Question 2)

In [4]:
import math

# ── Parameters ────────────────────────────────────────────────────────────────
S        = 40
K        = 40       # at-the-money
T        = 5/12
r        = 0.025
q        = 0.01
C_market = 2.75
tol      = 1e-6

# ── Helper functions ───────────────────────────────────────────────────────────
def norm_cdf(x):
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def norm_pdf(x):
    return math.exp(-0.5 * x**2) / math.sqrt(2 * math.pi)

def bs_call(sigma):
    """Black-Scholes call price (Merton model, continuous dividends)."""
    d1 = ((r - q + 0.5*sigma**2)*T) / (sigma*math.sqrt(T))
    d2 = d1 - sigma*math.sqrt(T)
    return S*math.exp(-q*T)*norm_cdf(d1) - K*math.exp(-r*T)*norm_cdf(d2)

def f(sigma):
    return bs_call(sigma) - C_market

def vega(sigma):
    d1 = ((r - q + 0.5*sigma**2)*T) / (sigma*math.sqrt(T))
    return S * math.exp(-q*T) * math.sqrt(T) * norm_pdf(d1)

# ── 1. BISECTION ──────────────────────────────────────────────────────────────
def bisection(a, b, tol=tol):
    print("=" * 65)
    print("BISECTION METHOD  [a=0.0001, b=1]")
    print("=" * 65)
    print(f"{'Iter':>4}  {'a':>12}  {'b':>12}  {'mid':>12}  {'f(mid)':>13}")
    print("-" * 58)

    for i in range(1000):
        mid = (a + b) / 2
        fm  = f(mid)
        print(f"{i+1:>4}  {a:>12.8f}  {b:>12.8f}  {mid:>12.8f}  {fm:>+13.8f}")
        if abs(b - a) < 2*tol:
            print(f"\nBisection:  sigma = {mid:.6f}  ({i+1} iterations)\n")
            return mid
        if f(a)*fm < 0:
            b = mid
        else:
            a = mid

# ── 2. SECANT ─────────────────────────────────────────────────────────────────
def secant(s0, s1, tol=tol):
    print("=" * 65)
    print("SECANT METHOD  [sigma_0=0.5, sigma_1=0.4]")
    print("=" * 65)
    print(f"{'Iter':>4}  {'sigma_n-1':>12}  {'sigma_n':>12}  {'f(sigma_n)':>13}")
    print("-" * 47)
    print(f"{0:>4}  {'—':>12}  {s0:>12.8f}  {f(s0):>+13.8f}")
    print(f"{1:>4}  {s0:>12.8f}  {s1:>12.8f}  {f(s1):>+13.8f}")

    for i in range(2, 1000):
        f0, f1 = f(s0), f(s1)
        s2 = s1 - f1*(s1 - s0)/(f1 - f0)   # secant update
        print(f"{i:>4}  {s1:>12.8f}  {s2:>12.8f}  {f(s2):>+13.8f}")
        if abs(s2 - s1) < tol:
            print(f"\nSecant:     sigma = {s2:.6f}  ({i} iterations)\n")
            return s2
        s0, s1 = s1, s2

# ── 3. NEWTON ─────────────────────────────────────────────────────────────────
def newton(sigma, tol=tol):
    print("=" * 65)
    print("NEWTON'S METHOD  [sigma_0=0.5]")
    print("=" * 65)
    print(f"{'Iter':>4}  {'sigma_n':>12}  {'C(sigma)':>12}  {'f(sigma)':>13}  {'Vega':>10}")
    print("-" * 57)

    for i in range(100):
        C  = bs_call(sigma)
        fv = f(sigma)
        v  = vega(sigma)
        print(f"{i:>4}  {sigma:>12.8f}  {C:>12.8f}  {fv:>+13.8f}  {v:>10.6f}")
        sigma_new = sigma - fv/v             # Newton update
        if abs(sigma_new - sigma) < tol:
            sigma = sigma_new
            print(f"{i+1:>4}  {sigma:>12.8f}  {bs_call(sigma):>12.8f}"
                  f"  {f(sigma):>+13.8f}  {vega(sigma):>10.6f}")
            print(f"\nNewton:     sigma = {sigma:.6f}  ({i+1} iterations)\n")
            return sigma
        sigma = sigma_new

# ── 4. APPROXIMATE FORMULA ────────────────────────────────────────────────────
def approx_formula(sigma_true):
    print("=" * 65)
    print("APPROXIMATE FORMULA  (part ii)")
    print("=" * 65)

    num   = math.sqrt(2*math.pi/T) * (C_market - (r - q)*T/2 * S)
    denom = S * (1 - (r + q)*T/2)
    sigma_approx = num / denom

    rel_error = abs(sigma_approx - sigma_true) / sigma_true

    print(f"  sqrt(2pi/T)             = {math.sqrt(2*math.pi/T):.6f}")
    print(f"  (r-q)*T/2 * S           = {(r-q)*T/2*S:.6f}")
    print(f"  C - (r-q)*T/2 * S       = {C_market - (r-q)*T/2*S:.6f}")
    print(f"  S*(1 - (r+q)*T/2)       = {denom:.6f}")
    print(f"\n  sigma_approx            = {sigma_approx:.6f}")
    print(f"  sigma_newton (true)     = {sigma_true:.6f}")
    print(f"  Relative error          = {rel_error:.6f}  ({rel_error*100:.4f}%)")

# ── Run ────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    bisection(0.0001, 1.0)
    secant(0.5, 0.4)
    sigma_newton = newton(0.5)
    approx_formula(sigma_newton)

BISECTION METHOD  [a=0.0001, b=1]
Iter             a             b           mid         f(mid)
----------------------------------------------------------
   1    0.00010000    1.00000000    0.50005000    +2.46636757
   2    0.00010000    0.50005000    0.25007500    -0.06953953
   3    0.25007500    0.50005000    0.37506250    +1.20135096
   4    0.25007500    0.37506250    0.31256875    +0.56648868
   5    0.25007500    0.31256875    0.28132187    +0.24859899
   6    0.25007500    0.28132187    0.26569844    +0.08955789
   7    0.25007500    0.26569844    0.25788672    +0.01001582
   8    0.25007500    0.25788672    0.25398086    -0.02976024
   9    0.25398086    0.25788672    0.25593379    -0.00987180
  10    0.25593379    0.25788672    0.25691025    +0.00007211
  11    0.25593379    0.25691025    0.25642202    -0.00489982
  12    0.25642202    0.25691025    0.25666614    -0.00241384
  13    0.25666614    0.25691025    0.25678820    -0.00117086
  14    0.25678820    0.25691025    0.2

### Appendix 3 (Question 3)

In [5]:
import math

# ── Parameters ────────────────────────────────────────────────────────────────
F      = 100        # face value
c      = 2.0        # semiannual coupon (4% annual / 2)
n      = 6          # number of periods (3 years × 2)
P_mkt  = 101.0      # market price
tol    = 1e-8

# Cash flows: coupon each period, face + coupon at maturity
CF      = [c] * n
CF[-1] += F

# ── Bond price and its derivatives w.r.t. semiannual yield y ──────────────────
def price(y):
    return sum(CF[t] / (1+y)**(t+1) for t in range(n))

def f(y):
    return price(y) - P_mkt

def dprice_dy(y):
    """First derivative dP/dy — used as f'(y) in Newton's method."""
    return sum(-(t+1)*CF[t] / (1+y)**(t+2) for t in range(n))

def d2price_dy2(y):
    """Second derivative d²P/dy² — used for convexity."""
    return sum((t+1)*(t+2)*CF[t] / (1+y)**(t+3) for t in range(n))

# ── Newton's method ────────────────────────────────────────────────────────────
def newton_yield(y0=0.02):
    print("=" * 65)
    print("NEWTON'S METHOD — semiannual yield")
    print("=" * 65)
    print(f"{'Iter':>4}  {'y_n':>14}  {'P(y_n)':>12}  {'f(y_n)':>13}  {'f prime':>12}")
    print("-" * 60)

    y = y0
    for i in range(100):
        P  = price(y)
        fv = f(y)
        dp = dprice_dy(y)
        print(f"{i:>4}  {y:>14.10f}  {P:>12.8f}  {fv:>+13.8f}  {dp:>12.6f}")
        y_new = y - fv / dp
        if abs(y_new - y) < tol:
            y = y_new
            print(f"{i+1:>4}  {y:>14.10f}  {price(y):>12.8f}"
                  f"  {f(y):>+13.8f}  {dprice_dy(y):>12.6f}")
            print(f"\nConverged in {i+1} iterations.")
            return y
        y = y_new

# ── Modified duration ─────────────────────────────────────────────────────────
def modified_duration(y):
    mac_periods = sum((t+1)*CF[t] / (1+y)**(t+1) for t in range(n)) / P_mkt
    mac_years   = mac_periods / 2           # convert periods → years
    mod_dur     = mac_years / (1 + y)       # standard formula
    return mac_periods, mac_years, mod_dur

# ── Convexity ─────────────────────────────────────────────────────────────────
def convexity(y):
    conv_semi   = d2price_dy2(y) / P_mkt   # in periods²
    conv_annual = conv_semi / 4             # annualise: divide by m² = 2² = 4
    return conv_semi, conv_annual

# ── PV table ──────────────────────────────────────────────────────────────────
def pv_table(y):
    print("\n" + "=" * 65)
    print("PV TABLE")
    print("=" * 65)
    print(f"{'Period':>6}  {'CF':>7}  {'PV(CF)':>10}  {'t·PV':>12}  {'t(t+1)·PV/(1+y)²':>18}")
    print("-" * 60)
    for t in range(n):
        pv   = CF[t] / (1+y)**(t+1)
        tpv  = (t+1) * pv
        t2pv = (t+1)*(t+2)*CF[t] / (1+y)**(t+3)
        print(f"{t+1:>6}  {CF[t]:>7.2f}  {pv:>10.6f}  {tpv:>12.6f}  {t2pv:>18.6f}")

# ── Run ────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    y_semi = newton_yield(y0=0.02)
    y_annual = 2 * y_semi

    mac_per, mac_yr, mod_dur = modified_duration(y_semi)
    conv_semi, conv_annual   = convexity(y_semi)

    pv_table(y_semi)

    print("\n" + "=" * 65)
    print("RESULTS")
    print("=" * 65)
    print(f"  Semiannual yield y         = {y_semi*100:.6f}%")
    print(f"  Annual yield (BEY)         = {y_annual*100:.6f}%")
    print(f"  Macaulay duration (periods)= {mac_per:.6f}")
    print(f"  Macaulay duration (years)  = {mac_yr:.6f}")
    print(f"  Modified duration          = {mod_dur:.6f} years")
    print(f"  Convexity                  = {conv_annual:.6f} years²")

NEWTON'S METHOD — semiannual yield
Iter             y_n        P(y_n)         f(y_n)       f prime
------------------------------------------------------------
   0    0.0200000000  100.00000000    -1.00000000   -560.143089
   1    0.0182147419  101.00605282    +0.00605282   -566.939717
   2    0.0182254182  101.00000022    +0.00000022   -566.898790
   3    0.0182254186  101.00000000    +0.00000000   -566.898789

Converged in 3 iterations.

PV TABLE
Period       CF      PV(CF)          t·PV    t(t+1)·PV/(1+y)²
------------------------------------------------------------
     1     2.00    1.964202      1.964202            3.789031
     2     2.00    1.929044      3.858088           11.163632
     3     2.00    1.894516      5.683547           21.927624
     4     2.00    1.860605      7.442421           35.891895
     5     2.00    1.827302      9.136510           52.874188
     6   102.00   91.524332    549.145989         3707.643655

RESULTS
  Semiannual yield y         = 1.822542%
 

### Appendix 4 (Question 4)

In [6]:
import math

# ── Parameters ────────────────────────────────────────────────────────────────
S     = 30
T     = 0.25        # 3 months
sigma = 0.30
q     = 0.01
r     = 0.025
tol   = 1e-6

# ── Helper functions ───────────────────────────────────────────────────────────
def norm_cdf(x):
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def norm_pdf(x):
    return math.exp(-0.5*x**2) / math.sqrt(2*math.pi)

def d1(K):
    return (math.log(S/K) + (r - q + 0.5*sigma**2)*T) / (sigma*math.sqrt(T))

def delta(K):
    """Call delta under continuous dividends: e^(-qT) * N(d1)."""
    return math.exp(-q*T) * norm_cdf(d1(K))

# ── g(K) and g'(K) for Newton's method ───────────────────────────────────────
def g(K):
    """Equation to solve: delta(K) - 0.5 = 0."""
    return delta(K) - 0.5

def dg_dK(K):
    """
    Derivative of g w.r.t. K via chain rule:
    dg/dK = e^(-qT) * phi(d1) * dd1/dK
    dd1/dK = -1 / (K * sigma * sqrt(T))
    => dg/dK = -e^(-qT) * phi(d1) / (K * sigma * sqrt(T))
    """
    return -math.exp(-q*T) * norm_pdf(d1(K)) / (K * sigma * math.sqrt(T))

# ── Newton's method ────────────────────────────────────────────────────────────
def newton_strike(K0=S):
    print("=" * 72)
    print("NEWTON'S METHOD — strike K such that Delta = 0.5")
    print("=" * 72)
    print(f"{'Iter':>4}  {'K_n':>14}  {'d1':>10}  {'Delta':>10}  {'g(K)':>13}  {'g prime':>12}")
    print("-" * 68)

    K = K0
    for i in range(100):
        d1v  = d1(K)
        dv   = delta(K)
        gv   = g(K)
        dgv  = dg_dK(K)
        print(f"{i:>4}  {K:>14.8f}  {d1v:>10.6f}  {dv:>10.6f}  {gv:>+13.8f}  {dgv:>12.8f}")
        K_new = K - gv / dgv
        if abs(K_new - K) < tol:
            K = K_new
            print(f"{i+1:>4}  {K:>14.8f}  {d1(K):>10.6f}  {delta(K):>10.6f}"
                  f"  {g(K):>+13.8f}  {dg_dK(K):>12.8f}")
            print(f"\nConverged in {i+1} iterations.")
            return K
        K = K_new

# ── Run ────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    K_star = newton_strike(K0=S)   # start ATM
    print(f"\nStrike K*   = {K_star:.6f}")
    print(f"Delta at K* = {delta(K_star):.8f}  (target = 0.5)")

NEWTON'S METHOD — strike K such that Delta = 0.5
Iter             K_n          d1       Delta           g(K)       g prime
--------------------------------------------------------------------
   0     30.00000000    0.100000    0.538480    +0.03847995   -0.08799142
   1     30.43731482    0.003520    0.500153    +0.00015250   -0.08716137
   2     30.43906446    0.003137    0.500000    +0.00000000   -0.08715647
   3     30.43906451    0.003137    0.500000    -0.00000000   -0.08715647

Converged in 3 iterations.

Strike K*   = 30.439065
Delta at K* = 0.50000000  (target = 0.5)


### Appendix 5 (Question 5)

In [3]:
import math

# Given data
sigma = 0.30
r = 0.03
tau = 0.5   # 6 months
q = 0.0     # non-dividend-paying

# Standard normal pdf and cdf
def phi(z):
    return math.exp(-z*z/2) / math.sqrt(2*math.pi)

def N(z):
    return 0.5 * (1 + math.erf(z / math.sqrt(2)))

# x = K/S
def d1(x):
    return (-math.log(x) + (r + 0.5*sigma**2)*tau) / (sigma*math.sqrt(tau))

def d2(x):
    return d1(x) - sigma*math.sqrt(tau)

# Theta/S, so sign is the same as Theta
def f(x):
    return -(sigma / (2*math.sqrt(tau))) * phi(d1(x)) \
           + r * x * math.exp(-r*tau) * N(-d2(x))

# Derivative of f(x)
def fp(x):
    term1 = -(d1(x) * phi(d1(x))) / (2 * tau * x)
    term2 = r * math.exp(-r*tau) * N(-d2(x))
    term3 = r * math.exp(-r*tau) * phi(d2(x)) / (sigma * math.sqrt(tau))
    return term1 + term2 + term3

# Newton's method
x = 1.30   # initial guess
for n in range(10):
    fx = f(x)
    fpx = fp(x)
    x_new = x - fx / fpx
    print(f"Iteration {n}: x = {x:.12f}, f(x) = {fx:.12e}")
    if abs(x_new - x) < 1e-12:
        x = x_new
        break
    x = x_new

print("\nRoot:")
print(f"K/S = {x:.12f}")
print(f"S/K = {1/x:.12f}")

Iteration 0: x = 1.300000000000, f(x) = -1.373888493372e-02
Iteration 1: x = 1.358026614752, f(x) = -6.428851817934e-04
Iteration 2: x = 1.361044475681, f(x) = -2.196117821728e-06
Iteration 3: x = 1.361054855842, f(x) = -2.616636768371e-11
Iteration 4: x = 1.361054855965, f(x) = 1.387778780781e-17

Root:
K/S = 1.361054855965
S/K = 0.734724243933


Appendix 6(Question 6
)

In [1]:
import math

# ---------- basic data ----------
overnight = 0.05
P05 = 97.5
P1  = 100
P3  = 102
P5  = 104

# ---------- 6-month zero rate ----------
z05 = -math.log(97.5/100)/0.5

# ---------- 1-year zero rate ----------
# 100 = 2.5*e^{-0.5 z05} + 102.5*e^{-1*z1}
z1 = -math.log((100 - 2.5*math.exp(-0.5*z05))/102.5)

# ---------- 3-year bond ----------
def z_1_to_3(t, z3):
    # linear interpolation between (1, z1) and (3, z3)
    return z1 + (t - 1.0)/2.0 * (z3 - z1)

def F3(z3):
    return (
        2.5*math.exp(-0.5*z05)
        + 2.5*math.exp(-1.0*z1)
        + 2.5*math.exp(-1.5*z_1_to_3(1.5, z3))
        + 2.5*math.exp(-2.0*z_1_to_3(2.0, z3))
        + 2.5*math.exp(-2.5*z_1_to_3(2.5, z3))
        + 102.5*math.exp(-3.0*z3)
        - 102
    )

def F3p(z3):
    total = 0.0
    for t in [1.5, 2.0, 2.5]:
        dzdz3 = (t - 1.0)/2.0
        total += -2.5 * t * dzdz3 * math.exp(-t * z_1_to_3(t, z3))
    total += -102.5 * 3.0 * math.exp(-3.0*z3)
    return total

x = 0.05
print("Newton iterations for 3-year bond:")
print(f"x0 = {x:.10f}")
k = 0
while True:
    x_new = x - F3(x)/F3p(x)
    k += 1
    print(f"x{k} = {x_new:.10f}")
    if abs(x_new - x) < 1e-6:
        x = x_new
        break
    x = x_new

z3 = x

# interpolated 1.5, 2, 2.5 year zero rates
z15 = z_1_to_3(1.5, z3)
z2  = z_1_to_3(2.0, z3)
z25 = z_1_to_3(2.5, z3)

# ---------- 5-year bond ----------
def z_3_to_5(t, z5):
    # linear interpolation between (3, z3) and (5, z5)
    return z3 + (t - 3.0)/2.0 * (z5 - z3)

def F5(z5):
    return (
        3*math.exp(-0.5*z05)
        + 3*math.exp(-1.0*z1)
        + 3*math.exp(-1.5*z15)
        + 3*math.exp(-2.0*z2)
        + 3*math.exp(-2.5*z25)
        + 3*math.exp(-3.0*z3)
        + 3*math.exp(-3.5*z_3_to_5(3.5, z5))
        + 3*math.exp(-4.0*z_3_to_5(4.0, z5))
        + 3*math.exp(-4.5*z_3_to_5(4.5, z5))
        + 103*math.exp(-5.0*z5)
        - 104
    )

def F5p(z5):
    total = 0.0
    for t in [3.5, 4.0, 4.5]:
        dzdz5 = (t - 3.0)/2.0
        total += -3 * t * dzdz5 * math.exp(-t * z_3_to_5(t, z5))
    total += -103 * 5.0 * math.exp(-5.0*z5)
    return total

y = 0.05
while True:
    y_new = y - F5(y)/F5p(y)
    if abs(y_new - y) < 1e-12:
        y = y_new
        break
    y = y_new

z5 = y
z35 = z_3_to_5(3.5, z5)
z4  = z_3_to_5(4.0, z5)
z45 = z_3_to_5(4.5, z5)

print("\nZero rates:")
print(f"overnight = {overnight:.6f}")
print(f"z(0.5)  = {z05:.6f}")
print(f"z(1.0)  = {z1:.6f}")
print(f"z(1.5)  = {z15:.6f}")
print(f"z(2.0)  = {z2:.6f}")
print(f"z(2.5)  = {z25:.6f}")
print(f"z(3.0)  = {z3:.6f}")
print(f"z(3.5)  = {z35:.6f}")
print(f"z(4.0)  = {z4:.6f}")
print(f"z(4.5)  = {z45:.6f}")
print(f"z(5.0)  = {z5:.6f}")

Newton iterations for 3-year bond:
x0 = 0.0500000000
x1 = 0.0420249912
x2 = 0.0421175911
x3 = 0.0421176038

Zero rates:
overnight = 0.050000
z(0.5)  = 0.050636
z(1.0)  = 0.049370
z(1.5)  = 0.047557
z(2.0)  = 0.045744
z(2.5)  = 0.043931
z(3.0)  = 0.042118
z(3.5)  = 0.044295
z(4.0)  = 0.046472
z(4.5)  = 0.048649
z(5.0)  = 0.050826


Appendix 9 (Question 9)

In [2]:
import math

# data
F = 100
coupon_rate = 0.08
y = 0.09
m = 4
n = 8

c = F * coupon_rate / m   # quarterly coupon
i = y / m

# price
cashflows = [c]*7 + [F+c]
B = sum(cashflows[k-1] / (1+i)**k for k in range(1, n+1))

# duration and convexity
D = (1/B) * sum((k/m) * cashflows[k-1] / (1+i)**(k+1) for k in range(1, n+1))
C = (1/B) * sum((k*(k+1)/m**2) * cashflows[k-1] / (1+i)**(k+2) for k in range(1, n+1))

print("B =", B)
print("D =", D)
print("C =", C)

dys = [0.001, 0.005, 0.01, 0.02, 0.04]

print("\nΔy        B_new,D     B_new,D,C   Actual Price   Err_D       Err_D,C")
for dy in dys:
    B_new_D = B * (1 - D*dy)
    B_new_DC = B * (1 - D*dy + 0.5*C*dy**2)

    i_new = (y + dy)/m
    B_actual = sum(cashflows[k-1] / (1+i_new)**k for k in range(1, n+1))

    err_D = abs(B_new_D - B_actual)/B_actual
    err_DC = abs(B_new_DC - B_actual)/B_actual

    print(f"{dy:<8.3f}  {B_new_D:>10.4f}  {B_new_DC:>10.4f}  {B_actual:>12.4f}  {err_D:>9.6f}  {err_DC:>9.6f}")

B = 98.18820384835145
D = 1.8254526399515039
C = 3.9232460429683442

Δy        B_new,D     B_new,D,C   Actual Price   Err_D       Err_D,C
0.001        98.0090     98.0092       98.0092   0.000002   0.000000
0.005        97.2920     97.2968       97.2968   0.000049   0.000000
0.010        96.3958     96.4151       96.4149   0.000198   0.000002
0.020        94.6034     94.6805       94.6793   0.000801   0.000013
0.040        91.0187     91.3269       91.3172   0.003269   0.000106


### Appendix 11 (Question 11)

In [3]:
import math

def exact(x):
    return (2 + x) * math.exp(-x)

def fd_solution(n):
    h = 1.0 / n
    x = [j * h for j in range(n + 1)]
    Y = [0.0] * (n + 1)

    # initial values from boundary conditions
    Y[0] = 2.0
    Y[1] = 2.0 - h

    # recurrence
    for j in range(1, n):
        Y[j+1] = ((2 - h*h) * Y[j] - (1 - h) * Y[j-1]) / (1 + h)

    return x, Y

print(f"{'n':>5} {'h':>10} {'Y(1)':>15} {'exact y(1)':>15} {'max error':>15}")
errors = []

for i in range(3, 7):
    n = 2**i
    h = 1.0 / n
    x, Y = fd_solution(n)

    exact_vals = [exact(xj) for xj in x]
    max_err = max(abs(Y[j] - exact_vals[j]) for j in range(n + 1))
    errors.append((n, max_err))

    print(f"{n:5d} {h:10.6f} {Y[-1]:15.10f} {exact(1):15.10f} {max_err:15.10e}")

print("\nObserved orders:")
for k in range(len(errors) - 1):
    n1, e1 = errors[k]
    n2, e2 = errors[k+1]
    p = math.log(e1 / e2, 2)
    print(f"from n={n1} to n={n2}: p = {p:.6f}")

    n          h            Y(1)      exact y(1)       max error
    8   0.125000    1.1024366775    1.1036383235 1.2016459945e-03
   16   0.062500    1.1033386858    1.1036383235 2.9963768630e-04
   32   0.031250    1.1035634622    1.1036383235 7.4861288099e-05
   64   0.015625    1.1036196112    1.1036383235 1.8712317259e-05

Observed orders:
from n=8 to n=16: p = 2.003721
from n=16 to n=32: p = 2.000927
from n=32 to n=64: p = 2.000232
